## Use AI embeddings to make Personal Song Recommendations

### Train the Model

#### 1.Loading the dataset containing the song playlists as well as each song’s metadata, such as its title and artist:

In [2]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [2]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

#### Train

In [3]:
from gensim.models import Word2Vec

# Train our Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

In [48]:
model.wv

In [4]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('3094', 0.9964907169342041),
 ('3167', 0.9961369633674622),
 ('1922', 0.9961000084877014),
 ('2849', 0.9960633516311646),
 ('5586', 0.9957617521286011),
 ('2014', 0.9957243800163269),
 ('2976', 0.9955792427062988),
 ('3126', 0.9952012896537781),
 ('3117', 0.9951426386833191),
 ('3105', 0.9944385886192322)]

In [6]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [9]:
s_id[0]

'3094'

In [15]:
all_reco= model.wv.most_similar(positive=str(song_id))
for s_id in all_reco[:5]:
    print("\n",songs_df.iloc[int(s_id[0])])


 title     Breaking The Law
artist        Judas Priest
Name: 3094 , dtype: object

 title     Unchained
artist    Van Halen
Name: 3167 , dtype: object

 title           One
artist    Metallica
Name: 1922 , dtype: object

 title     Run To The Hills
artist         Iron Maiden
Name: 2849 , dtype: object

 title     The Last In Line
artist                 Dio
Name: 5586 , dtype: object


### Find Songs similar to your taste

In [35]:
songs_df.shape

(75263, 2)

In [19]:
songs_df[songs_df["artist"].str.contains("kishore", na=False)]

,title,artist
id,,


In [23]:
songs_df[songs_df["artist"].str.contains("Sheeran", na=False)]

,title,artist
id,,
66161,Silver Bells,Phil Sheeran


In [37]:
songs_df[songs_df["artist"].str.contains("anyma", na=False)]

,title,artist
id,,


In [38]:
songs_df[songs_df["title"].str.contains("Shape of", na=False)]

,title,artist
id,,


In [41]:
songs_df[songs_df["title"].str.contains("Hypnotized", na=False)]

,title,artist
id,,
9630,Hypnotized,Fleetwood Mac
11330,Hypnotized (w\/ Akon),Plies
20092,Hypnotized,Linda Jones
21160,Hypnotized,Big Geminii


In [42]:
all_reco= model.wv.most_similar(positive=str(9630))
print("Seed Song:",songs_df.iloc[9630], "\n\nReco:")
for s_id in all_reco[:5]:
    print("\n",songs_df.iloc[int(s_id[0])])

Seed Song: title        Hypnotized
artist    Fleetwood Mac
Name: 9630 , dtype: object 

Reco:

 title     Rebel Rebel
artist    David Bowie
Name: 13308 , dtype: object

 title     Can't Find My Way Home
artist               Blind Faith
Name: 13114 , dtype: object

 title     Your Fool
artist    Will Hoge
Name: 13089 , dtype: object

 title     Getting Ready For Christmas Day
artist                         Paul Simon
Name: 55165 , dtype: object

 title         Blue Skies Again
artist    Jessica Lea Mayfield
Name: 26601 , dtype: object


In [28]:
songs_df[songs_df["artist"].str.contains("Bryan Adams", na=False)]

,title,artist
id,,
3419,Heaven,Bryan Adams
3564,Run To You,Bryan Adams
3670,Summer Of '69,Bryan Adams
3743,Somebody,Bryan Adams
3977,Cuts Like A Knife,Bryan Adams
8365,(Everything I Do) I Do It For You,Bryan Adams
8455,Straight From The Heart,Bryan Adams
8500,Have You Ever Really Loved A Woman?,Bryan Adams
8515,Please Forgive Me,Bryan Adams


In [33]:
all_reco= model.wv.most_similar(positive=str(3670))
print("Seed Song:",songs_df.iloc[3670], "\n\nReco:")
for s_id in all_reco[:5]:
    print("\n",songs_df.iloc[int(s_id[0])])

Seed Song: title     Summer Of '69
artist      Bryan Adams
Name: 3670 , dtype: object 

Reco:

 title     Hurts So Good
artist      John Cougar
Name: 3820 , dtype: object

 title     What About Love
artist              Heart
Name: 4052 , dtype: object

 title     Do You Believe In Love
artist     Huey Lewis & The News
Name: 3762 , dtype: object

 title         Sussudio
artist    Phil Collins
Name: 3457 , dtype: object

 title         Footloose
artist    Kenny Loggins
Name: 3633 , dtype: object


In [45]:
songs_df[songs_df["title"].str.contains("Shut Up", na=False)]

,title,artist
id,,
2570,Shut Up And Kiss Me,Mary Chapin Carpenter
15616,"Shut Up (w\/ Duece Poppito, Trina & Co)",Trick Daddy
15801,Shut Up I Am Dreaming Of Places Where Lovers H...,Sunset Rubdown
18478,Shut Up And Drive,Widespread Panic
30952,Shut Up & Drive,Rihanna
52903,Shut Up!,Simple Plan
53883,Shut Up Looking At Me,Overnight Lows
73783,Shut Up And Let Me Go,The Ting Tings


In [46]:
all_reco= model.wv.most_similar(positive=str(2570))
print("Seed Song:",songs_df.iloc[2570], "\n\nReco:")
for s_id in all_reco[:5]:
    print("\n",songs_df.iloc[int(s_id[0])])

Seed Song: title       Shut Up And Kiss Me
artist    Mary Chapin Carpenter
Name: 2570 , dtype: object 

Reco:

 title              Here
artist    Rascal Flatts
Name: 6589 , dtype: object

 title        Maybe Tonight
artist    Margaret Durante
Name: 27241 , dtype: object

 title     She's Every Woman
artist         Garth Brooks
Name: 18204 , dtype: object

 title     I Still Believe In That
artist                 Ash Bowers
Name: 27240 , dtype: object

 title     Electric Car (w\/ Robin Goldwasser)
artist                   They Might Be Giants
Name: 19756 , dtype: object


## Use a different Encoder Model all-mpnet-base-v2

In [1]:
import pandas as pd
from urllib import request
from sentence_transformers import SentenceTransformer, util
import torch

2026-02-15 18:53:39.833622: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-15 18:53:39.850743: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771181619.870335   17044 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771181619.876573   17044 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-15 18:53:39.897376: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [3]:
#PREPARE TEXT FOR MPNET ---

songs_df['combined_text'] = songs_df['title'] + " by " + songs_df['artist']

# LOAD MPNET & ENCODE ---

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# We only encode unique songs that actually appear in our metadata
song_descriptions = songs_df['combined_text'].tolist()
song_ids = songs_df.index.tolist()

# This creates the "brain" of your recommender
embeddings = model.encode(song_descriptions, convert_to_tensor=True, show_progress_bar=True)

/opt/conda/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/2352 [00:00<?, ?it/s]

In [7]:
# import gc
# import torch
# torch.cuda.empty_cache()
# gc.collect()

0

In [17]:
embeddings

tensor([[-0.0319, -0.0384,  0.0168,  ..., -0.0092, -0.0057, -0.0192],
        [-0.0377, -0.0552,  0.0040,  ..., -0.0448, -0.0508, -0.0361],
        [-0.0781, -0.0426, -0.0221,  ..., -0.0364,  0.0007, -0.0014],
        ...,
        [-0.0718, -0.0066, -0.0199,  ..., -0.0023,  0.0079, -0.0716],
        [-0.0131, -0.0257, -0.0190,  ..., -0.0302, -0.0347, -0.0425],
        [-0.0232,  0.0515, -0.0024,  ...,  0.0256, -0.0703, -0.0090]],
       device='cuda:0')

In [16]:
songs_df.iloc[3670]

title                           Summer Of '69
artist                            Bryan Adams
combined_text    Summer Of '69 by Bryan Adams
Name: 3670 , dtype: object

In [36]:
# 4. --- PREDICT SIMILAR SONGS ---
target_song_id = 3670 # Summer of 69


target_embedding = embeddings[target_song_id]

# Calculate cosine similarity between target and all other songs
cosine_scores = util.cos_sim(target_embedding, embeddings)[0]

# Get top 5 matches (excluding the song itself)
top_results = torch.topk(cosine_scores, k=6)

In [48]:
print(f"Songs similar to: {songs_df.iloc[target_song_id]['combined_text']}\n")
for score, idx in zip(top_results.values[1:], top_results.indices[1:]):
    # similar_song_id = song_ids[idx]
    print(f"Score: {score:.4f} | {songs_df.iloc[int(idx)]['combined_text']}")

Songs similar to: Summer Of '69 by Bryan Adams

Score: 0.7931 | Summer '68 by Pink Floyd
Score: 0.7657 | That Summer by Garth Brooks
Score: 0.7303 | Summer Day by Sheryl Crow
Score: 0.7289 | Long Hot Summer by Keith Urban
Score: 0.7273 | Summertime by George Benson


##  Use a different Encoder Model - Google GLove (Transfer Learning)
Should you do this?
For a song recommender, usually no.
GloVe was trained on Wikipedia articles; it knows what "Kishore Kumar" means in a sentence, but it doesn't know which songs "belong" together in a playlist.
Your Original Word2Vec is actually better for this specific task because it learns the "vibe" of a playlist based on song co-occurrence, which Wikipedia doesn't track. 

In [53]:
import gensim.downloader as api
from gensim.models import Word2Vec
import numpy as np

# 1. Load the pre-trained vectors (read-only)
g_model = api.load("glove-wiki-gigaword-50")

# 2. Initialize a fresh Word2Vec model with the SAME vector_size
# We don't train it yet; we just build the vocabulary from your playlists
new_model = Word2Vec(vector_size=50, window=20, min_count=1)
new_model.build_vocab(playlists)

# 3. Intersect (Inject) GloVe weights into your new model
# This fills your model with Wikipedia's knowledge where words match
new_model.wv.vectors_lockf = np.ones(len(new_model.wv)) # Allow weights to be updated
new_model.wv.intersect_word2vec_format(api.load("glove-wiki-gigaword-50", return_path=True))

# 4. NOW train on your playlists
new_model.train(playlists, total_examples=new_model.corpus_count, epochs=10)

(18873948, 18879330)

In [55]:
all_reco= new_model.wv.most_similar(positive=str(3670))
print("Seed Song:",songs_df.iloc[3670], "\n\nReco:")
for s_id in all_reco[:5]:
    print("\n",songs_df.iloc[int(s_id[0])])

Seed Song: title                           Summer Of '69
artist                            Bryan Adams
combined_text    Summer Of '69 by Bryan Adams
Name: 3670 , dtype: object 

Reco:

 title                 -
artist                -
combined_text    - by -
Name: 33189 , dtype: object

 title                    Me Fui
artist                     Bebe
combined_text    Me Fui by Bebe
Name: 14030 , dtype: object

 title                          Beat Dat Beat
artist                            DJ Pauly D
combined_text    Beat Dat Beat by DJ Pauly D
Name: 73631 , dtype: object

 title                                               Caldonia
artist                       Willie Nelson & Wynton Marsalis
combined_text    Caldonia by Willie Nelson & Wynton Marsalis
Name: 44352 , dtype: object

 title                              Fraud In The '80s
artist                                Mates Of State
combined_text    Fraud In The '80s by Mates Of State
Name: 24218 , dtype: object


In [56]:
all_reco= new_model.wv.most_similar(positive=str(2570))
print("Seed Song:",songs_df.iloc[2570], "\n\nReco:")
for s_id in all_reco[:5]:
    print("\n",songs_df.iloc[int(s_id[0])])

Seed Song: title                                     Shut Up And Kiss Me
artist                                  Mary Chapin Carpenter
combined_text    Shut Up And Kiss Me by Mary Chapin Carpenter
Name: 2570 , dtype: object 

Reco:

 title                          Again
artist                    Vic Damone
combined_text    Again by Vic Damone
Name: 24692 , dtype: object

 title                             Everybody Needs Somebody To Love
artist                                              Wilson Pickett
combined_text    Everybody Needs Somebody To Love by Wilson Pic...
Name: 73012 , dtype: object

 title                       Grape Juice City
artist                               Ratatat
combined_text    Grape Juice City by Ratatat
Name: 29956 , dtype: object

 title                             La Carta Que Nunca Envie
artist                                       Los Palominos
combined_text    La Carta Que Nunca Envie by Los Palominos
Name: 29712 , dtype: object

 title                 